# Preparación para tablero en Metabase

In [1]:
import geopandas as gdp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
# Cargamos los archivos que veniamos usando
datos_censales_con_gdf = gdp.read_file("../data/processed/datos_censales_con_gdf.gpkg")

In [3]:
datos_censales_con_gdf.columns

Index(['NOMPROV', 'PROV', 'NOMDEPTO', 'DEPTO', 'FRAC', 'RADIO', 'TIPO', 'LINK',
       'OBS2020', 'CodigoFraccion', 'CodigoRadioCensal', 'NBI_Hacinamiento_Si',
       'NBI_Hacinamiento_No', 'NBI_Vivienda_Si', 'NBI_Vivienda_No',
       'NBI_Sanitarias_Si', 'NBI_Sanitarias_No', 'NBI_Escolaridad_Si',
       'NBI_Escolaridad_No', 'NBI_Subsistencia_Si', 'NBI_Subsistencia_No',
       'NBI_Si', 'NBI_No', 'Sin_IPMH', 'IPMH_Solo_Recursos_Corrientes',
       'IPMH_Solo_Recursos_Patrimoniales', 'IPMH_Convergente', 'Total_Hogares',
       'Total_Poblacion', 'hogares_en_casa_a', 'hogares_en_casa_b',
       'hogares_en_rancho', 'hogares_en_casilla', 'hogares_en_departamento',
       'hogares_en_pieza_inquilinato', 'hogares_en_local_no_hab',
       'hogares_en_vivienda_movil', 'viv_casa_a', 'viv_casa_b', 'viv_rancho',
       'viv_casilla', 'viv_departamento', 'viv_pieza_inquilinato',
       'viv_local_no_hab', 'viv_vivienda_movil', 'Total_Viviendas',
       'tasa_NBI_%', 'NBI_Si_tasa_categorizado', '

In [4]:
# --- 1. INCIDENCIA (Magnitud total) ---
# Sumamos todos los que tienen ALGO de privación y dividimos por total de hogares
# Fórmula: (RC + RP + CO) / Total_Hogares * 100
datos_censales_con_gdf['tasa_incidencia_IPMH_%'] = (
    (datos_censales_con_gdf['IPMH_Solo_Recursos_Corrientes'] + 
     datos_censales_con_gdf['IPMH_Solo_Recursos_Patrimoniales'] + 
     datos_censales_con_gdf['IPMH_Convergente']) 
    / datos_censales_con_gdf['Total_Hogares']
) * 100

# --- 2. INTENSIDAD (Gravedad) ---
# Cuánto representa el "Convergente" sobre el total de pobres.
# Primero calculamos el denominador (Total de Hogares con Privación)
total_privados = (
    datos_censales_con_gdf['IPMH_Solo_Recursos_Corrientes'] + 
    datos_censales_con_gdf['IPMH_Solo_Recursos_Patrimoniales'] + 
    datos_censales_con_gdf['IPMH_Convergente']
)

# Fórmula: (Convergente / Total_Privados) * 100
datos_censales_con_gdf['Tasa_Intensidad_IPMH'] = (datos_censales_con_gdf['IPMH_Convergente'] / total_privados) * 100

# --- 3. PREVALENCIA (RPRC - Composición) ---
# Relación entre problemas de Ingresos vs Problemas de Vivienda
# Numerador: Todos los que tienen problemas de Recursos (Solo RC + Convergente)
# Denominador: Todos los que tienen problemas Patrimoniales (Solo RP + Convergente)

num_recursos = datos_censales_con_gdf['IPMH_Solo_Recursos_Corrientes'] + datos_censales_con_gdf['IPMH_Convergente']
den_patrimonial = datos_censales_con_gdf['IPMH_Solo_Recursos_Patrimoniales'] + datos_censales_con_gdf['IPMH_Convergente']

# Fórmula: (Num / Den) * 100
# Reemplazamos el 0 en el denominador por 1 para que no de error (infinity)
datos_censales_con_gdf['Prevalencia_IPMH'] = (
    num_recursos / den_patrimonial.replace(0, 1)
) * 100

print("Cálculo de indicadores derivados completado.")

Cálculo de indicadores derivados completado.


In [ ]:
# Expoertar el MAPA (Solo la forma y el ID)
# Usamos la columna 'LINK' como identificador único
mapa_metabase = datos_censales_con_gdf[['LINK', 'geometry']]
mapa_metabase.to_file("mapa_radios_rosario.geojson", driver='GeoJSON')
print("1. mapa_radios_rosario.geojson (Cargar en Metabase Admin -> Maps)")

1. mapa_radios_rosario.geojson (Cargar en Metabase Admin -> Maps)


In [ ]:
id_vars = [
    'LINK', 
    'RADIO', 
    'Total_Poblacion', 
    'Total_Hogares',
    'Total_Viviendas'
    # 'Nombre_Barrio',  <-- AGREGAR 
    # 'Distrito'    <-- AGREGAR
]

value_vars = [
    #  BLOQUE NBI OFICIAL
    'tasa_NBI_%',
    'tasa_NBI_hacinamiento_%',
    'tasa_NBI_vivienda_%',             
    'tasa_NBI_sanitarias_%',
    'tasa_NBI_escolaridad_%',
    'tasa_NBI_subsistencia_%',
        
    #  BLOQUE IPMH (Dimensiones) 
    'tasa_Sin_IPMH_%',
    'tasa_Solo_Corriente_%',
    'tasa_Solo_Patrimonial_%',
    'tasa_Convergente_%',
    
    #  BLOQUE IPMH (Indicadores derivados) 
    'tasa_incidencia_IPMH_%',          # (RC + RP + CO) / Total
    'Prevalencia_IPMH',                # Ratio Ingresos / Vivienda
    'Tasa_Intensidad_IPMH',            # Gravedad Convergente
    
    #  VARIABLES DE CONTROL 
    'tasa_hogares_unipersonales_%', 
    'tasa_viviendas_inconvenientes_%',
    'tasa_viviendas_adecuadas_%'
]

#  Hacemos el melt para convertir de formato ancho a formato largo
df_long = datos_censales_con_gdf.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='Tipo_Indicador',  # Nombre de la nueva columna de categorías
    value_name='Valor'     # Nombre de la nueva columna con los números
)

# Guardamos los datos en un csv
print(df_long.head())
df_long.to_csv("../data/processed/datos_censo_formato_largo_v1.csv", index=False)

        LINK RADIO  Total_Poblacion  Total_Hogares  Total_Viviendas  \
0  820840301    01              805            267              262   
1  820840302    02              920            285              284   
2  820840303    03              736            240              238   
3  820840304    04             1316            439              431   
4  820840305    05              653            243              240   

  Tipo_Indicador      Valor  
0     tasa_NBI_%  19.475655  
1     tasa_NBI_%   1.754386  
2     tasa_NBI_%   6.666667  
3     tasa_NBI_%   5.239180  
4     tasa_NBI_%   5.349794  


In [ ]:
df_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24576 entries, 0 to 24575
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   LINK             24576 non-null  object 
 1   RADIO            24576 non-null  object 
 2   Total_Poblacion  24576 non-null  int64  
 3   Total_Hogares    24576 non-null  int64  
 4   Total_Viviendas  24576 non-null  int64  
 5   Tipo_Indicador   24576 non-null  object 
 6   Valor            24575 non-null  float64
dtypes: float64(1), int64(3), object(3)
memory usage: 1.3+ MB


-----------------------------------

Volvemos a correr, pero ahora agregando nuevas variables para segunda parte del tablero

In [ ]:
# Agregamos los indicadores derivados al formato ancho para que estén disponibles en el tablero
datos_censales_con_gdf['tasa_incidencia_IPMH_%'] = (
    (datos_censales_con_gdf['IPMH_Solo_Recursos_Corrientes'] + 
     datos_censales_con_gdf['IPMH_Solo_Recursos_Patrimoniales'] + 
     datos_censales_con_gdf['IPMH_Convergente']) 
    / datos_censales_con_gdf['Total_Hogares']
) * 100

total_privados = (
    datos_censales_con_gdf['IPMH_Solo_Recursos_Corrientes'] + 
    datos_censales_con_gdf['IPMH_Solo_Recursos_Patrimoniales'] + 
    datos_censales_con_gdf['IPMH_Convergente']
)
datos_censales_con_gdf['Tasa_Intensidad_IPMH'] = (datos_censales_con_gdf['IPMH_Convergente'] / total_privados) * 100

num_recursos = datos_censales_con_gdf['IPMH_Solo_Recursos_Corrientes'] + datos_censales_con_gdf['IPMH_Convergente']
den_patrimonial = datos_censales_con_gdf['IPMH_Solo_Recursos_Patrimoniales'] + datos_censales_con_gdf['IPMH_Convergente']
datos_censales_con_gdf['Prevalencia_IPMH'] = (num_recursos / den_patrimonial.replace(0, 1)) * 100

print("Cálculo de indicadores derivados completado.")


Cálculo de indicadores derivados completado.
mapa_radios_rosario.geojson listo para Metabase.


In [ ]:
# 1. Definimos qué columnas se quedan quietas (Id variables)
id_vars = [
    'LINK', 
    'RADIO', 
    'Total_Poblacion', 
    'Total_Hogares',
    'Total_Viviendas',
    'viv_casa_a', 
    'viv_casa_b', 
    'viv_rancho',       
    'viv_casilla', 
    'viv_departamento', 
    'viv_pieza_inquilinato',       
    'viv_local_no_hab', 
    'viv_vivienda_movil'
]

# 2. VARIABLES DINÁMICAS (Las que van a ir al filtro del mapa)
value_vars = [
    # --- BLOQUE NBI OFICIAL ---
    'tasa_NBI_%',
    'tasa_NBI_hacinamiento_%',
    'tasa_NBI_vivienda_%',             
    'tasa_NBI_sanitarias_%',
    'tasa_NBI_escolaridad_%',
    'tasa_NBI_subsistencia_%',
        
    # --- BLOQUE IPMH (Dimensiones) ---
    'tasa_Sin_IPMH_%',
    'tasa_Solo_Corriente_%',
    'tasa_Solo_Patrimonial_%',
    'tasa_Convergente_%',
    
    # --- BLOQUE IPMH (Indicadores derivados) ---
    'tasa_incidencia_IPMH_%',          
    'Prevalencia_IPMH',                
    'Tasa_Intensidad_IPMH',            
    
    # --- VARIABLES DE CONTROL ---
    'tasa_hogares_unipersonales_%', 
    'tasa_viviendas_inconvenientes_%', 
    'tasa_viviendas_adecuadas_%'
]

# 3. Hacemos el MELT (La magia)
# Solo derretimos las variables de porcentaje. Los totales absolutos quedan en cada fila.
df_long = datos_censales_con_gdf.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='tipo_indicador',  
    value_name='valor'     
)

# 4. Verificamos y Guardamos
print(df_long.head())
df_long.to_csv("../data/processed/datos_censo_formato_largo.csv", index=False)

        LINK RADIO  Total_Poblacion  Total_Hogares  Total_Viviendas  \
0  820840301    01              805            267              262   
1  820840302    02              920            285              284   
2  820840303    03              736            240              238   
3  820840304    04             1316            439              431   
4  820840305    05              653            243              240   

   viv_casa_a  viv_casa_b  viv_rancho  viv_casilla  viv_departamento  \
0          87         139          20            9                 2   
1         273           3           0            0                 7   
2         202          21           2            3                 7   
3         371          40           0            5                14   
4         198          26           3            2                10   

   viv_pieza_inquilinato  viv_local_no_hab  viv_vivienda_movil tipo_indicador  \
0                      3                 1                 